# Lab 48: The Tickle Paradox

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 6, *The Embodied Avatar*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-48.ipynb)

**What you will do:** rate how ticklish self-touch, other-touch and delayed self-touch feel, check the pattern against the book's forward-model account, and run a numerical forward model that predicts and cancels its own self-caused signal.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 48 you tried to tickle your own palm or the sole of your own foot, then compared this
with being tickled by a friend and with a delayed or friend-guided version of your own touch.
Most people feel almost nothing from pure self-touch, an intense tickle from someone else's
touch, and something in between from the delayed or guided version. The book explains this with
a forward model: the brain predicts the exact sensory consequences of a movement it commands,
and subtracts that prediction from the incoming signal, leaving nothing to feel when the two
match exactly.

## Record your results

Rate how ticklish or surprising each version of the test felt, from 0 (no sensation at all) to
10 (extremely ticklish). Repeat each version two or three times -- a single rating is noisy.

In [ ]:
trials = pd.DataFrame({
    "condition": ["self"] * 3 + ["other"] * 3 + ["delayed/guided self"] * 3,
    "trial": [1, 2, 3] * 3,
    "rating (0-10)": [1, 0, 1, 8, 7, 9, 4, 5, 3],   # example data: replace with your own ratings
})
display(trials)
summary = trials.groupby("condition")["rating (0-10)"].agg(["mean", "std"])
display(summary)
summary["mean"].plot.barh(xerr=summary["std"], color=["firebrick", "grey", "steelblue"], figsize=(6, 2.4))
plt.xlabel("ticklishness rating (0-10)"); plt.xlim(0, 10); plt.show()

## Compare

The book does not give a numeric threshold for how ticklish self-touch should feel --
Blakemore et al. (1998) reported the effect using a robotic arm and a variable delay, not a
rating scale. What it does claim is a clear pattern: predictable self-touch should feel weaker
than touch from someone else, and adding delay or unpredictability to self-touch should partly
restore the tickle. Check whether your own numbers show that pattern.

In [ ]:
self_mean = summary.loc["self", "mean"]
other_mean = summary.loc["other", "mean"]
delayed_mean = summary.loc["delayed/guided self", "mean"]

print(f"self: {self_mean:.1f}, other: {other_mean:.1f}, delayed/guided self: {delayed_mean:.1f}")
if self_mean < other_mean:
    print("Matches the book's claim: predictable self-touch feels less ticklish than touch from someone else.")
else:
    print("Does not match: check whether you were really unable to predict your own touch, or whether you")
    print("rated leniently out of habit.")
if self_mean < delayed_mean <= other_mean:
    print("Also matches: adding delay or guidance moved self-touch partway back towards other-touch, as the")
    print("forward-model account predicts.")
elif delayed_mean <= self_mean:
    print("The delayed/guided version did not feel more ticklish than plain self-touch for you -- the delay")
    print("or guidance may not have been unpredictable enough to break the forward model's prediction.")

## The AI side

The tickle paradox is really about prediction and subtraction: a forward model predicts a
self-caused signal, and the prediction is subtracted from what actually arrives, leaving only
the unpredicted part. Below, a toy forward model predicts a signal it "caused" itself and
cancels it, while an externally caused pulse survives the cancellation -- a numerical version
of the trick the book attributes to the cerebellum.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
t = np.arange(0, 2, 0.001)   # 2 seconds, sampled at 1 kHz
self_caused_signal = np.exp(-0.5 * ((t - 1.0) / 0.02) ** 2)         # a brief self-produced touch pulse
external_signal = 0.6 * np.exp(-0.5 * ((t - 1.5) / 0.02) ** 2)      # a brief externally caused touch pulse
raw_input = self_caused_signal + external_signal + np.random.normal(0, 0.02, t.shape)

# The forward model's prediction: a noisy but fairly accurate copy of the self-caused signal only,
# generated from the motor command "at t=1.0s" -- it knows nothing about the external touch.
predicted = np.exp(-0.5 * ((t - 1.0) / 0.02) ** 2) + np.random.normal(0, 0.01, t.shape)

residual = np.clip(raw_input - predicted, 0, None)   # what the forward-model account says reaches awareness

fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
axes[0].plot(t, raw_input, color="grey"); axes[0].set_ylabel("raw touch\nsignal")
axes[1].plot(t, predicted, color="steelblue"); axes[1].set_ylabel("forward-model\nprediction")
axes[2].plot(t, residual, color="firebrick"); axes[2].set_ylabel("residual\n(what you'd notice)")
axes[2].set_xlabel("time (s)")
for ax in axes:
    ax.axvline(1.0, color="grey", linestyle=":", alpha=0.5)
    ax.axvline(1.5, color="grey", linestyle=":", alpha=0.5)
plt.tight_layout(); plt.show()

print(f"Peak residual at the self-caused touch (t=1.0s): {residual[(t > 0.99) & (t < 1.01)].max():.3f}")
print(f"Peak residual at the externally caused touch (t=1.5s): {residual[(t > 1.49) & (t < 1.51)].max():.3f}")

The self-caused pulse at t = 1.0 s is almost entirely cancelled, while the externally caused
pulse at t = 1.5 s survives -- the model "feels" the second touch but not the first, just as
you feel a friend's tickle but not your own. The analogy has limits: this toy model is told in
advance exactly when its own signal will arrive, whereas the cerebellum builds that prediction
from an efference copy of the actual motor command, continuously, for movements it has never
been timed on before. It also has no cerebellum-like structure, no learning, and no notion of
"ticklishness" as opposed to any other kind of subtracted signal.

## Questions to think about

1. Your forward-model prediction above was generated with less noise (0.01) than the real touch (0.02). What happens to the residual at t=1.0s if you increase the prediction noise to 0.05? Does the self-caused touch become as noticeable as the external one?
2. The lab says a friend guiding your own hand to tickle you still reduces the tickle, even though someone else is involved. In the forward-model story, why would your own motor involvement matter more than whose hand is doing the touching?
3. The book mentions that people with cerebellar damage can sometimes tickle themselves. In the code above, what would happen to the residual if the forward model's prediction were set to zero at every time point, as if no prediction were being made at all? Try it.
4. Comedy is described as a case where prediction error is pleasurable rather than aversive. Using the forward-model idea, what do you think happens to a joke's "prediction error" the second time you hear it, and why might that make it stop being funny?

## Challenge

Devise your own delay-based version of the tickle test: get a friend to lightly tap the back of
your hand a fixed number of times, then, with your eyes closed, reproduce the same tapping
pattern on your own other hand -- either instantly or after silently counting to five first.
Rate the ticklishness of your self-produced version at each delay.

In [ ]:
# Example data: replace with your own ratings after each self-tapping trial.
delay_s = [0.0, 1.0, 2.0, 5.0]     # seconds between the friend's tap and your own copy
my_ticklishness = [1, 2, 4, 6]     # example data: 0 (nothing) to 10 (very ticklish)

results = pd.DataFrame({"delay (s)": delay_s, "ticklishness (0-10)": my_ticklishness})
display(results)
results.plot.line(x="delay (s)", y="ticklishness (0-10)", marker="o", legend=False, figsize=(6, 3))
plt.ylim(0, 10); plt.ylabel("ticklishness (0-10)"); plt.show()
print("If ticklishness rises with delay, your forward model was predicting the touch precisely when you")
print("reproduced it immediately, and losing precision as the delay grew -- the same mechanism Blakemore")
print("et al. (1998) demonstrated with a robotic arm.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-48.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")